# Harness Engineering Demo: Better Harness Beats Bigger Model


**Thesis:** model quality matters, but harness maturity determines whether an AI workflow is production-ready. A medium model in a strong harness can outperform a stronger model in a no-harness or weak-harness setup because production quality depends on governed context, tools, validation, memory, safety gates, observability, and repair loops.

We will show the spectrum:

1. **Great model + no harness**: incident ticket only, no controlled guides, sensors, or steering loop.
2. **Great model + weak harness**: multi-agent scratchpad exists, but it is untyped, unprovenanced, and not governed by sensors or repair.
3. **Medium model + SDK harness**: Strands-style abstraction for tools, hooks, memory, and multi-agent orchestration.
4. **Provider plug-and-play harness**: provider-specific harness lane, represented by DeepSeek adapter boundary.

The reliable part of the demo is deterministic. The live model section calls Ollama Cloud so management can see how this connects to real model backends.

## Demo Architecture

```text
Colab notebook
      ↓
Harness demo repo + Python SDKs
      ↓
Scenario: multi-agent incident response
      ↓
Scorecard: evidence, runbook, safety, memory, completeness
      ↓
Optional live calls to Ollama Cloud
```

Colab is the runtime. Ollama Cloud is the model backend.

## 1. Clone The Repo

Replace `REPO_URL` with your GitHub URL after pushing the project.

If you already uploaded this notebook into the cloned repo, skip this cell and `%cd` into the repo folder.

In [ ]:
# Replace this with your pushed GitHub repository URL.
REPO_URL = "https://github.com/YOUR_ORG/ollama-harness-engineering-demo.git"
REPO_DIR = "ollama-harness-engineering-demo"

from pathlib import Path
import os

# If we are not already inside the repo, clone it or move into an existing clone.
if not Path("pyproject.toml").exists():
    if Path(REPO_DIR).exists():
        os.chdir(REPO_DIR)
    else:
        if "YOUR_ORG" in REPO_URL:
            raise ValueError(
                "Replace REPO_URL with your GitHub repo URL, then rerun this cell. "
                "Example: https://github.com/my-org/ollama-harness-engineering-demo.git"
            )
        !git clone $REPO_URL
        os.chdir(REPO_DIR)

print("Current directory:", Path.cwd())
print("Project files:")
!ls -la

assert Path("requirements.txt").exists(), "requirements.txt not found. You are not inside the repo."
assert Path("pyproject.toml").exists(), "pyproject.toml not found. You are not inside the repo."

## 2. Install The Demo Dependencies

This happens inside Colab, so it does not depend on your office Mac allowing Python packages.

The important libraries are:

- `strands-agents`: SDK-level harness abstraction.
- `ollama`: direct calls to Ollama Cloud.
- `openai`: useful for OpenAI-compatible endpoints.
- `typer` and `rich`: CLI and readable scorecards.
- `pytest`: quick health check.

In [ ]:
from pathlib import Path

assert Path("requirements.txt").exists(), "Run the clone/%cd setup cell first. requirements.txt is missing here."
assert Path("pyproject.toml").exists(), "Run the clone/%cd setup cell first. pyproject.toml is missing here."

!pip install -r requirements.txt
!pip install -e .

## 3. Import The Repo Harness

Colab is only the presentation surface. The actual use case and harness workflow are imported from the repo.

No static lane outputs are used in this notebook. The management demo path below calls Ollama Cloud live and scores the actual model-generated artifacts.

# Optional HITL Gate: When The Harness Should Stop

Human-in-the-loop is a natural extension of this incident workflow.

The harness should escalate when an output still has safety risks, missing runbook obligations, or low confidence after the goal loop budget is exhausted. The point is not to ask a human to read every model answer; the point is to ask a human only when deterministic sensors say the workflow is outside the approved operating envelope.

In this use case, HITL would be triggered by examples like:

- forbidden action appears: drop/truncate promotion cache table
- payment-write disablement suggested before the 12% for 5 minutes threshold
- rollback plan missing after goal-loop attempts
- unsupported operational systems invented, such as CloudWatch/RDS/Grafana when not present in the scenario



In [ ]:
def hitl_decision_packet(result):
    findings = evaluate_rules(scenario, result)
    escalation_items = [
        finding for finding in findings
        if finding.severity in {"miss", "risk"}
    ]
    if result.score >= 85 and not escalation_items:
        return "No HITL escalation needed. Harness result is ready for normal human review."

    lines = [
        "## HITL Escalation Packet",
        "",
        f"Lane: `{result.lane.value}`",
        f"Score: `{result.score}/100`",
        "",
        "Human decision needed because deterministic sensors still found:",
    ]
    for item in escalation_items:
        lines.append(f"- **{item.title}** ({item.category}): {item.detail}")
    lines.extend([
        "",
        "Recommended human action:",
        "- approve as-is only with explicit incident-commander signoff",
        "- otherwise send back through repair with the listed findings",
        "- for forbidden actions, require a runbook exception or incident commander approval",
    ])
    return "\n".join(lines)

# Example: run this against any lane after it executes.
# display(Markdown(hitl_decision_packet(live_harness)))
# display(Markdown(hitl_decision_packet(strands_result)))
# display(Markdown(hitl_decision_packet(deepseek_scored)))


In [ ]:
from pathlib import Path
from pprint import pprint
import sys

repo_src = str(Path.cwd() / "src")
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)

assert Path("src/harness_demo").exists(), "Not in repo root or src/harness_demo is missing."

from IPython.display import HTML, Markdown, display

from harness_demo.live import run_live_hand_built_lane, run_live_raw_lane, run_live_weak_harness_lane, score_freeform_answer
from harness_demo.rules import evaluate_rules
from harness_demo.summarizer import critique_groundedness_with_ollama, summarize_findings_with_ollama, summarize_root_cause_for_management
from harness_demo.colab_display import (
    render_comparison_markdown,
    render_hallucination_review_markdown,
    render_management_summary_markdown,
    render_executive_findings_markdown,
    render_memory_markdown,
    render_model_output_html,
    render_result_markdown,
    render_rule_findings_markdown,
)
from harness_demo.scenarios import load_incident_scenario
from harness_demo.domain import Lane

scenario = load_incident_scenario("incident-response")

print("Loaded scenario:", scenario.id)
print("Scenario name:", scenario.name)


## 4. The Use Case: Incident Response, Not Prompt Comparison

The scenario is a production checkout incident after a promotion launch.

A management audience should see the difference between:

- a model answering from a ticket alone
- an AI workflow that uses controlled evidence, approved runbooks, shared memory, reviewer checks, and repair

This is the point of harness engineering: the system around the model turns an answer into a governed workflow result.

In [ ]:
print("INCIDENT TICKET")
pprint(scenario.incident)

print("\nQUALITY CONTRACT USED BY THE SCORER")
pprint(scenario.expected)

## 5. Controlled Context Available To The Harness

These are not pasted wholesale into the raw model call.

They are available to the **harnessed workflow** through separate controlled steps:

- log investigator sees logs
- runbook agent sees runbook
- memory agent sees prior incident memory
- planner sees the shared memory created by earlier agents
- reviewer sees the final plan and checks policy/completeness

That separation is the harness. It is what changes between the weak and strong setup.

In [ ]:
print("LOG TOOL DATA")
print(scenario.logs)

print("RUNBOOK TOOL DATA")
print(scenario.runbook)

print("PRIOR MEMORY TOOL DATA")
print(scenario.prior_memory)

## 6. Three Different Things: No Harness, Weak Harness, Strong Harness

We should be precise. Missing memory sharing is not a weak harness. It is mostly **no harness**.

For this demo we use three levels:

| Level | What it means in this incident workflow |
| --- | --- |
| No harness | One model call against the incident ticket. No explicit guides, no tools, no sensors, no steering loop. |
| Weak harness | Multi-agent workflow with a shared scratchpad. It has some feedforward context and memory sharing, but memory is untyped and unprovenanced, and there are no independent sensors before the final plan, no reviewer gate, and no repair loop. |
| Strong harness | Work is decomposed into controlled steps with explicit guides, tool-scoped context, shared memory, deterministic sensors, reviewer checks, and repair. |

This is closer to the harness-engineering framing: a harness is a system of **feedforward guides**, **feedback sensors**, and a **steering/self-correction loop**. A weak harness has only some of these pieces, or has them in a non-operational form.

## 7. What Changes In The Strong Harness?

The strong harness is not merely a longer prompt. It changes the execution environment around the model.

| Harness component | Strong harness behavior in this demo |
| --- | --- |
| Feedforward guides | Each agent gets a narrow role and only the context it is supposed to use. |
| Controlled tools | Logs, runbook, and prior memory are separate inputs to separate steps, not one undifferentiated paste. |
| Shared state | Agent outputs are accumulated into `SharedMemory`, which later steps consume. |
| Computational sensors | The scorer checks evidence, runbook usage, safety, memory usage, and completeness. |
| Reviewer gate | The final plan is checked for forbidden actions and missing required fields. |
| Steering loop | If the reviewer finds issues, the repair step asks the model to revise against concrete objections. |

The medium model does not win because it was prompted more nicely. It gets a more governable operating environment.

## 8. Configure Ollama Cloud

Ollama Cloud is the live backend. The repo harness is the application workflow.

Do not hardcode the key in the notebook.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OLLAMA_API_KEY"):
    os.environ["OLLAMA_API_KEY"] = getpass("Enter OLLAMA_API_KEY: ")

print("OLLAMA_API_KEY configured:", bool(os.environ.get("OLLAMA_API_KEY")))

## 9. Choose Models

The intended comparison:

- **strong model + no harness**: bare incident ticket only
- **strong model + weak harness**: multi-agent scratchpad and context, but memory is untyped/unprovenanced and there are no sensors or repair
- **medium model + strong harness**: decomposed workflow with guides, controlled context, memory, sensors, reviewer, and repair
- **strong model + strong harness**: same mature harness with a stronger model to show the additional upside

Change these model names based on your Ollama Cloud subscription.

In [ ]:
NO_HARNESS_MODEL = "gpt-oss:120b"
WEAK_HARNESS_MODEL = "gpt-oss:120b"
STRONG_HARNESS_MODEL = "gpt-oss:20b"
STRONG_MODEL_STRONG_HARNESS_MODEL = "gpt-oss:120b"

print("Strong model, no harness:", NO_HARNESS_MODEL)
print("Strong model, weak harness:", WEAK_HARNESS_MODEL)
print("Medium model, strong harness:", STRONG_HARNESS_MODEL)

## Optional Summarizer Model

The deterministic rule engine decides truth. This model only rewrites findings into a management-friendly summary.

Use a cheaper/simple model here if available.

In [ ]:
SUMMARY_MODEL = "gpt-oss:20b"
print("Summary/polish model:", SUMMARY_MODEL)

## Two Kinds Of Evaluation

The score remains deterministic. It checks exact contract items: required evidence, required runbook steps, forbidden actions, memory use, and required final-plan fields.

The qualitative groundedness critique is different. It uses an evaluator LLM to catch semantic inventions that string rules can miss: invented teams, tools, dashboards, timelines, unsupported mitigations, or overconfident language. This critique explains risk, but it does not change the numeric score.



# Live Run 1: Strong Model + No Harness

This cell makes one live Ollama Cloud call.

The model receives the incident ticket only. It does not receive logs, runbook, prior memory, output contract, reviewer, or repair loop.

This is the **no-harness baseline**, not a weak multi-agent harness.

In [ ]:
live_no_harness = run_live_raw_lane(scenario, model_name=NO_HARNESS_MODEL)
display(Markdown(render_management_summary_markdown(live_no_harness)))
display(Markdown(render_executive_findings_markdown(live_no_harness)))
display(Markdown(render_rule_findings_markdown(scenario, live_no_harness)))
display(HTML(render_model_output_html("Actual model output", live_no_harness.final_answer)))
display(Markdown(render_memory_markdown(live_no_harness.memory)))
raw_groundedness_critique = critique_groundedness_with_ollama(
    scenario=scenario,
    result=live_no_harness,
    model_name=SUMMARY_MODEL,
)
display(Markdown("## LLM Qualitative Groundedness Critique"))
display(Markdown(raw_groundedness_critique))


# Live Run 2: Strong Model + Weak Harness

This is the actual **weak harness** comparison.

The strong model runs in a weak multi-agent harness. There is shared memory, but it is only a plain scratchpad.

That is better than no harness because agents can pass context forward. But it is still weak because:

- the scratchpad has no schema
- memory entries have no source provenance
- the workflow does not validate intermediate notes
- the final plan is not reviewed before scoring
- there is no repair loop

This is the realistic corporate distinction: shared memory alone is not harness engineering. Governed memory plus sensors and steering is.

In [ ]:
live_weak = run_live_weak_harness_lane(scenario, model_name=WEAK_HARNESS_MODEL)
display(Markdown(render_management_summary_markdown(live_weak)))
display(Markdown(render_executive_findings_markdown(live_weak)))
display(Markdown(render_rule_findings_markdown(scenario, live_weak)))
display(HTML(render_model_output_html("Actual model output", live_weak.final_answer)))
display(Markdown(render_memory_markdown(live_weak.memory)))


# Live Run 3: Medium Model + Strong Harness

This cell calls Ollama Cloud multiple times through the repo harness workflow.

Each call has a specific role and a controlled context boundary:

1. log investigator: incident + logs
2. runbook agent: incident + runbook
3. memory agent: incident + prior memory
4. planner: shared memory + required output fields + forbidden actions
5. reviewer: deterministic safety/completeness checks
6. repair: only if reviewer finds issues

The repair loop is not last-answer-wins. It scores each candidate in isolation, gives safety precedence, keeps the best accepted candidate, and records worse repair attempts only in the audit trail.

The score is computed from the actual generated agent outputs and final plan.

In [ ]:
live_harness = run_live_hand_built_lane(scenario, model_name=STRONG_HARNESS_MODEL)
display(Markdown(render_management_summary_markdown(live_harness)))
display(Markdown(render_executive_findings_markdown(live_harness)))
display(Markdown(render_rule_findings_markdown(scenario, live_harness)))
display(HTML(render_model_output_html("Actual agent outputs", live_harness.final_answer)))


## 10. Inspect The Shared Memory

This is the harness value in concrete form.

The workflow did not just produce prose. It produced operational state that can be checked, audited, and reused.

In [ ]:
display(Markdown(render_memory_markdown(live_harness.memory)))


# Live Run 4: Strong Model + Strong Harness

This cell runs the same strong harness with the stronger model.

Why this matters for management:

- It separates **harness value** from **model value**.
- If medium+strong harness performs close to strong+strong harness, the harness is doing real work.
- If strong+strong harness improves further, the story becomes: model upgrades help most when the harness is already mature.

This is not the primary thesis lane; it is the scope-of-upside lane.


In [ ]:
live_strong_model_harness = run_live_hand_built_lane(scenario, model_name=STRONG_MODEL_STRONG_HARNESS_MODEL)
display(Markdown(render_management_summary_markdown(live_strong_model_harness)))
display(Markdown(render_executive_findings_markdown(live_strong_model_harness)))
display(Markdown(render_rule_findings_markdown(scenario, live_strong_model_harness)))
display(HTML(render_model_output_html("Actual strong-model harness agent outputs", live_strong_model_harness.final_answer)))


In [ ]:
display(Markdown(render_memory_markdown(live_strong_model_harness.memory)))


## 11. Live Side-By-Side Result

This is the only scorecard to show as evidence.

All rows come from live Ollama Cloud calls. The difference is the harness maturity, not a hardcoded result.

In [ ]:
display(Markdown(render_comparison_markdown([live_no_harness, live_weak, live_harness, live_strong_model_harness])))


## 12. LLM-Polished Management Summaries

These summaries are generated by an Ollama Cloud model, but the model is **not judging**.

Inputs to the summarizer:

- deterministic score
- deterministic pass/fail checks
- deterministic rule findings
- extracted shared memory

The summarizer is only used to make the findings easier to present.

In [ ]:
for label, result in [
    ("No harness", live_no_harness),
    ("Weak harness", live_weak),
    ("Medium model + strong harness", live_harness),
    ("Strong model + strong harness", live_strong_model_harness),
]:
    findings = evaluate_rules(scenario, result)
    summary = summarize_findings_with_ollama(
        scenario=scenario,
        result=result,
        findings=findings,
        model_name=SUMMARY_MODEL,
    )
    management_root_cause = summarize_root_cause_for_management(
        scenario=scenario,
        result=result,
        findings=findings,
        model_name=SUMMARY_MODEL,
    )
    display(Markdown(f"# {label}: Management-Language Root Cause"))
    display(Markdown(management_root_cause))
    critique = critique_groundedness_with_ollama(
        scenario=scenario,
        result=result,
        model_name=SUMMARY_MODEL,
    )
    display(Markdown(f"# {label}: LLM-Polished Rule Summary"))
    display(Markdown(summary))
    display(Markdown(f"# {label}: Qualitative Groundedness Critique"))
    display(Markdown(critique))


## 12. Explain The Difference In Plain English

Use this in the meeting after the live scorecard:

- The no-harness lane asks a strong model to improvise from the ticket alone.
- The weak-harness lane has multi-agent shared memory, but the memory is an ungoverned scratchpad with no schema, provenance, sensors, or steering.
- The strong-harness lane decomposes the work, gives each agent controlled context, accumulates shared memory, and checks the final result against policy.
- The scorecard is not grading writing style. It is grading production requirements: evidence, runbook use, safety, memory, completeness.

That is harness engineering.

# Optional Smoke Test

Only use this for your own pre-demo sanity check, not as a management proof.

```bash
harness-demo compare --scenario incident-response
python -m pytest -p no:cacheprovider
```

Those commands validate the repo wiring. The live evidence is above.

# Industry Spectrum: Harness Engineering Is Getting Productized

After the live evidence, show management that this is not only our custom pattern. The industry is moving from hand-built harnesses toward SDKs and plug-and-play harness runtimes.

Important boundary:

- The **proof** in this notebook is the live Ollama Cloud no/weak/strong harness comparison above.
- The **industry movement** section below explains why teams should expect more of these controls to become reusable infrastructure.

## Strands Agents: SDK-Level Harness Abstraction

Strands is the SDK-level part of the story.

It shows that common harness capabilities are becoming framework features:

| Harness need | Strands direction |
| --- | --- |
| Tool boundaries | `@tool` definitions and tool registries |
| Feedforward guides | agent instructions, tool descriptions, steering handlers |
| Feedback sensors | hooks before/after tool calls |
| Context management | conversation managers and summarization |
| Safety controls | guardrails/hooks that block or redirect actions |
| Observability | traces and hook-level inspection |
| Multi-agent workflows | agent-as-tool and swarm-style patterns |

Management message:

> We hand-built the harness to make the mechanics visible. SDKs like Strands show these mechanics are becoming reusable developer infrastructure.

Source: https://strandsagents.com/

In [ ]:
# Lightweight sanity check: Strands is installed in the Colab environment.
# This is not the live proof. It shows the SDK is available for the next implementation layer.
import strands
print("Strands SDK import ok:", strands.__name__)

## DeepSeek Harness: Plug-And-Play Harness Runtime

DeepSeek Harness is the plug-and-play runtime part of the story.

Its public developer-preview positioning is **“Everything is a plugin.”** The harness runtime composes capabilities such as:

- models
- tools
- skills
- sessions
- sandboxes
- storage
- loops
- scheduling
- UI

It also emphasizes traceability: what the model sees, tool calls/results, context injection, and subagent scheduling are recorded in an append-only session log.

Management message:

> DeepSeek Harness shows where the market is heading: harness capabilities are becoming runtime infrastructure that teams can compose instead of rebuilding from scratch.

Source: https://deepseek.com/harness/en/

## How To Say This Without Overselling

Use this wording:

> Today we proved the harness value with our live hand-built workflow. Strands and DeepSeek Harness show that the same ideas are being productized: SDKs are packaging tools, hooks, memory, and observability; harness runtimes are packaging models, tools, sessions, loops, and traceability as plugins.

Avoid this wording:

> Strands/DeepSeek produced the same scores in this notebook.

We should only make that claim after wiring them into live execution.

## Communication Harness: Correct Answer, Right Audience

Production harnesses do not only check factual correctness. They can also regulate how the answer is communicated.

For this incident demo, the communication policy is:

- explain root cause in management language
- do not overpromise certainty or timelines
- keep customer/business impact visible
- do not hide unresolved safety, runbook, or groundedness risks
- avoid unnecessary implementation jargon unless it is explained

In the custom path, this is handled by `summarize_root_cause_for_management`. In the Strands path, we also show the SDK-style version with `LLMSteeringHandler`, which critiques and guides communication quality without becoming the factual judge.



# Adoption Experiment 1: Strands Multi-Agent SDK Harness

This is the “team adoption” section, now structured as a fairer multi-agent comparison.

Goal: show that the controls we hand-built are becoming SDK-level concepts: separate agents, scoped tools, explicit handoffs, trace attributes, and model-provider abstraction.

This cell attempts a real Strands multi-agent run with the same incident scenario:

1. **Log agent** sees only logs.
2. **Runbook agent** sees only the approved runbook.
3. **Memory agent** sees only prior incident memory.
4. **Planner agent** receives the shared handoff from the specialist agents and writes the final plan.
5. The repo deterministic evaluator scores the final answer.

This is still not magic. Strands supplies SDK primitives; our harness contract still defines the roles, handoffs, and scoring gates.

Sources: https://strandsagents.com/docs/user-guide/concepts/tools/ and https://strandsagents.com/docs/user-guide/concepts/multi-agent/agents-as-tools/


In [ ]:
STRANDS_MODEL = "gpt-oss:20b"  # adjust if your Strands provider expects a different model id
print("Strands experiment model:", STRANDS_MODEL)

In [ ]:
strands_result = None
strands_scored_output = ""
strands_audit_transcript = ""
strands_repair_attempts = []
try:
    import os
    import json
    from strands import Agent, tool
    from strands.models.ollama import OllamaModel
    try:
        from strands.vended_plugins.goal import GoalLoop
    except Exception:
        GoalLoop = None

    if not os.environ.get("OLLAMA_API_KEY"):
        raise RuntimeError("OLLAMA_API_KEY is required for the Strands Ollama Cloud lane.")

    strands_model = OllamaModel(
        host="https://ollama.com",
        model_id=STRANDS_MODEL,
        ollama_client_args={
            "headers": {"Authorization": "Bearer " + os.environ["OLLAMA_API_KEY"]}
        },
        temperature=0.2,
    )

    @tool
    def get_checkout_logs() -> str:
        """Return checkout incident logs."""
        return scenario.logs

    @tool
    def get_checkout_runbook() -> str:
        """Return the approved checkout promotion incident runbook."""
        return scenario.runbook

    @tool
    def get_prior_incident_memory() -> str:
        """Return prior similar incident memory."""
        return scenario.prior_memory

    log_agent = Agent(
        model=strands_model,
        tools=[get_checkout_logs],
        system_prompt=(
            "You are the log investigator agent. Use only get_checkout_logs. "
            "Return evidence for likely cause and downstream symptoms. Do not invent tools or dashboards."
        ),
        trace_attributes={"demo": "harness-engineering", "lane": "strands-sdk", "agent": "log-investigator"},
    )
    runbook_agent = Agent(
        model=strands_model,
        tools=[get_checkout_runbook],
        system_prompt=(
            "You are the runbook agent. Use only get_checkout_runbook. "
            "Return approved mitigation steps and safety constraints. Do not add unapproved operations."
        ),
        trace_attributes={"demo": "harness-engineering", "lane": "strands-sdk", "agent": "runbook"},
    )
    memory_agent = Agent(
        model=strands_model,
        tools=[get_prior_incident_memory],
        system_prompt=(
            "You are the memory agent. Use only get_prior_incident_memory. "
            "Return prior lessons relevant to this incident."
        ),
        trace_attributes={"demo": "harness-engineering", "lane": "strands-sdk", "agent": "memory"},
    )

    log_output = str(log_agent(
        "Incident ticket:\n" + scenario.incident["prompt"] + "\n\nCall the log tool and return grounded evidence only."
    ))
    runbook_output = str(runbook_agent(
        "Incident ticket:\n" + scenario.incident["prompt"] + "\n\nCall the runbook tool and return approved actions and constraints only."
    ))
    memory_output = str(memory_agent(
        "Incident ticket:\n" + scenario.incident["prompt"] + "\n\nCall the prior memory tool and return relevant lessons only."
    ))

    strands_shared_memory = {
        "incident": scenario.incident,
        "log_agent_evidence": log_output,
        "runbook_agent_constraints": runbook_output,
        "memory_agent_lessons": memory_output,
        "required_output_fields": scenario.expected["required_final_plan_fields"],
        "required_evidence": scenario.expected["required_evidence"],
        "required_runbook_steps": scenario.expected["required_runbook_steps"],
        "forbidden_actions": scenario.expected["forbidden_actions"],
    }

    strands_goal_loop = None
    if GoalLoop is not None:
        strands_goal_loop = GoalLoop(
            goal=(
                "Return a complete incident plan with likely_cause, evidence, safe_next_action, "
                "rollback_plan, customer_impact, and open_questions. It must include the approved rollback "
                "to previous promotion configuration, must keep payment writes enabled unless the 12% for 5 minutes "
                "threshold is met, must use 60 seconds TTL during rollout, and must not invent tools, owners, "
                "dashboards, or operational telemetry."
            ),
            max_attempts=3,
        )

    planner_agent = Agent(
        model=strands_model,
        plugins=[strands_goal_loop] if strands_goal_loop else [],
        system_prompt=(
            "You are the planner agent in a Strands multi-agent harness. "
            "Use the specialist handoff exactly. Return JSON or concise markdown with likely_cause, "
            "evidence, safe_next_action, rollback_plan, customer_impact, and open_questions. "
            "Do not invent tools, owners, dashboards, thresholds, or operational facts that are not in the handoff."
        ),
        trace_attributes={"demo": "harness-engineering", "lane": "strands-sdk", "agent": "planner"},
    )
    planner_output = str(planner_agent(json.dumps(strands_shared_memory, indent=2)))
    goal_loop_report = "GoalLoop not available in this Strands runtime."
    if strands_goal_loop is not None:
        try:
            goal_loop_report = str(strands_goal_loop.last_result(planner_agent))
        except Exception as goal_exc:
            goal_loop_report = "GoalLoop result could not be read: " + repr(goal_exc)

    strands_base_transcript = "\n\n".join([
        "## Strands Log Agent",
        log_output,
        "## Strands Runbook Agent",
        runbook_output,
        "## Strands Memory Agent",
        memory_output,
    ])
    strands_scored_output = "\n\n".join([
        strands_base_transcript,
        "## Strands Planner Agent",
        planner_output,
        "## Strands GoalLoop Result",
        goal_loop_report,
    ])
    strands_audit_transcript = strands_scored_output

    strands_result = score_freeform_answer(
        scenario=scenario,
        answer=strands_scored_output,
        lane=Lane.STRANDS_SDK,
        title="Strands multi-agent SDK harness experiment",
        takeaway=(
            "Strands now runs separate specialist agents with scoped tools and explicit handoffs. "
            "The repo deterministic evaluator still decides whether the output is production-ready."
        ),
        used_harness_memory=True,
    )

    # Deterministic app-level goal loop: Strands GoalLoop can improve the response,
    # but our production contract is still the repo rule engine.
    strands_repair_attempts = []
    repair_output = ""
    strands_best_result = strands_result
    strands_best_output = strands_scored_output
    strands_best_rank = ((1 if strands_result.checks.get("safety") else 0), strands_result.score)
    for repair_attempt in range(1, 4):
        if all(strands_best_result.checks.values()):
            break
        strands_findings_for_repair = evaluate_rules(scenario, strands_best_result)
        repair_agent = Agent(
            model=strands_model,
            system_prompt=(
                "You are the repair agent in a Strands multi-agent production harness. "
                "Revise the final plan to satisfy deterministic reviewer findings. "
                "Use only the specialist handoff, approved runbook steps, and required fields. "
                "Return JSON only. Do not invent tools, owners, dashboards, thresholds, or operational facts."
            ),
            trace_attributes={"demo": "harness-engineering", "lane": "strands-sdk", "agent": "repair"},
        )
        repair_payload = {
            "deterministic_findings": [finding.__dict__ for finding in strands_findings_for_repair],
            "specialist_handoff": strands_shared_memory,
            "current_accepted_output": strands_best_output,
            "required_output_fields": scenario.expected["required_final_plan_fields"],
            "required_evidence": scenario.expected["required_evidence"],
            "required_runbook_steps": scenario.expected["required_runbook_steps"],
            "forbidden_actions": scenario.expected["forbidden_actions"],
        }
        repair_output = str(repair_agent(json.dumps(repair_payload, indent=2)))
        candidate_output = "\n\n".join([
            strands_base_transcript,
            f"## Strands Current Repair Candidate {repair_attempt}",
            repair_output,
        ])
        candidate_result = score_freeform_answer(
            scenario=scenario,
            answer=candidate_output,
            lane=Lane.STRANDS_SDK,
            title="Strands multi-agent SDK harness experiment with goal loop",
            takeaway=(
                "Strands runs specialist agents and SDK GoalLoop; the repo deterministic goal loop still decides "
                "whether the output satisfies the production contract."
            ),
            used_harness_memory=True,
        )
        candidate_rank = ((1 if candidate_result.checks.get("safety") else 0), candidate_result.score)
        accepted = candidate_rank > strands_best_rank
        if accepted:
            strands_best_result = candidate_result
            strands_best_output = candidate_output
            strands_best_rank = candidate_rank
        strands_repair_attempts.append({
            "attempt": repair_attempt,
            "score": candidate_result.score,
            "checks": candidate_result.checks,
            "accepted": accepted,
            "misses": [f.__dict__ for f in evaluate_rules(scenario, candidate_result) if f.severity in {"miss", "risk"}],
        })
        strands_audit_transcript += f"\n\n## Strands Deterministic Repair Attempt {repair_attempt}\n{repair_output}"

    if strands_repair_attempts:
        strands_audit_transcript += "\n\n## Deterministic Goal Loop Attempts\n" + json.dumps(strands_repair_attempts, indent=2)
        strands_result = strands_best_result
        strands_scored_output = strands_best_output

    display(Markdown(render_management_summary_markdown(strands_result)))
    display(Markdown(render_executive_findings_markdown(strands_result)))
    display(Markdown(render_rule_findings_markdown(scenario, strands_result)))
    display(HTML(render_model_output_html("Strands scored final output", strands_result.final_answer)))
    display(HTML(render_model_output_html("Strands audit transcript, including earlier failed attempts", strands_audit_transcript)))
except Exception as exc:
    display(Markdown(f"""
## Strands Multi-Agent Experiment Did Not Produce A Scorable Output

This does **not** invalidate harness engineering. It means the Strands provider/runtime configuration needs more setup in this Colab environment.

**Error type:** `{type(exc).__name__}`  
**Error:** `{exc}`

What to try next:

- confirm `strands.models.ollama.OllamaModel` supports Ollama Cloud auth headers in this version
- confirm `OLLAMA_API_KEY` is set
- confirm `STRANDS_MODEL` is available in your Ollama Cloud subscription
- keep the hand-built harness as the proof and use Strands as the SDK adoption path
"""))


In [ ]:
if strands_result is not None:
    strands_findings = evaluate_rules(scenario, strands_result)
    strands_summary = summarize_findings_with_ollama(
        scenario=scenario,
        result=strands_result,
        findings=strands_findings,
        model_name=SUMMARY_MODEL,
    )
    strands_management_root_cause = summarize_root_cause_for_management(
        scenario=scenario,
        result=strands_result,
        findings=strands_findings,
        model_name=SUMMARY_MODEL,
    )
    strands_critique = critique_groundedness_with_ollama(
        scenario=scenario,
        result=strands_result,
        model_name=SUMMARY_MODEL,
    )
    display(Markdown("# Strands: Management-Language Root Cause"))
    display(Markdown(strands_management_root_cause))
    display(Markdown("# Strands: LLM-Polished Rule Summary"))
    display(Markdown(strands_summary))
    display(Markdown("# Strands: Qualitative Groundedness Critique"))
    display(Markdown(strands_critique))
else:
    display(Markdown("_No Strands summary generated because there was no scorable Strands output._"))


## Strands Communication Steering

This cell shows a different harness facet: not correctness scoring, but response quality for the audience.

The deterministic evaluator remains the judge. The steering handler acts like a communication reviewer: it pushes the agent toward plain management language, no overpromising, explicit customer impact, and visible residual risk.



In [ ]:
strands_tone_summary = None
if strands_result is not None:
    try:
        from strands import Agent
        from strands.vended_plugins.steering import LLMSteeringHandler

        class ExecutiveToneGuardrailHandler(LLMSteeringHandler):
            name = "executive-tone-guardrail"

            def __init__(self):
                super().__init__(
                    system_prompt=(
                        "Evaluate the incident summary against these communication policies: "
                        "1. Explain root cause in management language. "
                        "2. Do not overpromise timelines or certainty. "
                        "3. Acknowledge customer/business impact. "
                        "4. Do not hide safety, runbook, or groundedness risks. "
                        "5. Keep it concise and avoid unnecessary technical jargon. "
                        "If violated, provide specific guidance on what to fix."
                    )
                )

        tone_handler = ExecutiveToneGuardrailHandler()
        communication_agent = Agent(
            model=strands_model,
            plugins=[tone_handler],
            system_prompt=(
                "You are the management communication agent in a production incident harness. "
                "Translate the deterministic evaluation into executive language. "
                "Do not change scores, pass/fail values, facts, or risks. "
                "Treat current_final_findings as the accepted plan assessment. "
                "Treat audit_only_rejected_attempts as history only; do not say those actions are in the accepted final plan. "
                "If any check is false, do not call the result Pass."
            ),
            trace_attributes={"demo": "harness-engineering", "lane": "strands-sdk", "agent": "communication"},
        )
        communication_payload = {
            "score": strands_result.score,
            "checks": strands_result.checks,
            "status": "PASS" if all(strands_result.checks.values()) else "NEEDS REVIEW / REPAIR",
            "shared_memory": strands_result.memory.__dict__,
            "accepted_final_output": strands_scored_output,
            "current_final_findings": [finding.__dict__ for finding in evaluate_rules(scenario, strands_result)],
            "audit_only_rejected_attempts": [attempt for attempt in strands_repair_attempts if not attempt.get("accepted")],
        }
        strands_tone_summary = str(communication_agent(json.dumps(communication_payload, indent=2, default=str)))
        display(Markdown("# Strands: Communication-Steered Management Summary"))
        display(Markdown(strands_tone_summary))
    except Exception as exc:
        display(Markdown(f"""
## Strands Communication Steering Not Available

The core Strands lane is still valid. This optional communication-control cell could not run in the current Strands runtime.

**Error type:** `{type(exc).__name__}`  
**Error:** `{exc}`
"""))
else:
    display(Markdown("_No Strands communication steering generated because there was no scorable Strands output._"))



# Adoption Experiment 2: DeepSeek Harness Headless + Ollama

DeepSeek Harness is the plug-and-play runtime part of the story, but in Colab we use it **headless**, not through the Web UI.

The working Colab path is:

1. Install/check Node, Ollama CLI, and DSH prerequisites from notebook cells.
2. Start `ollama serve` in the single Colab terminal and leave it running.
3. Run `ollama launch dsh --model ... --config` once so Ollama writes DSH model settings.
4. Copy those settings into `~/.dsh/settings.yaml`, because direct headless DSH reads DSH settings, not the Ollama launcher settings path.
5. Run direct `npx --yes @deepseek-ai/dsh --profile headless ...` for the smoke test and incident task.

Why headless:

- The Web UI is a local shell/code agent surface; tunneling it out of Colab is a security/compliance risk.
- The headless profile runs one bounded task, prints a final answer, and exits.
- That lets us score the final answer with the same deterministic evaluator used for the other lanes.

For fairness, this lane must be **multi-agentic**, not just a single DSH task. So this section does two things:

1. Runs a small blocking-subagent smoke test.
2. Runs the incident task only after asking the lead DSH agent to delegate evidence review and safety review to separate blocking subagents.

Important preview caveat: reports in the DSH community indicate that headless + background/continuable subagents can lose child results. For the demo we therefore ask for **blocking** subagent calls, not background delegation.

What this proves if it runs:

- Ollama is the model/provider boundary.
- DeepSeek Harness is a packaged harness/runtime boundary.
- DSH can delegate to child agents in headless mode and return their findings to the parent.
- The same deterministic scoring can evaluate its final output without trusting its own self-assessment.

What this does **not** prove yet:

- It does not prove our custom role-specialized incident workflow and DSH have identical internals.
- It does not prove long-running/background subagent behavior in Colab.

So the management-safe framing is: **DeepSeek Harness is the packaged runtime adoption signal; our custom and Strands lanes remain the controlled harness comparison.**

Sources:

- https://docs.ollama.com/integrations/deepseek-harness
- https://github.com/deepseek-ai/deepseek-harness/blob/master/apps/cli/reference/README.md
- https://github.com/deepseek-ai/deepseek-harness/blob/master/docs/subsystems/subagent.md


In [ ]:
# DeepSeek Harness runtime setup for fresh Colab.
# This cell is intentionally runnable. It installs prerequisites if they are missing.

from IPython.display import Markdown, display

display(Markdown("""
## DeepSeek Harness Runtime Setup

Run this in a fresh Colab runtime before starting the DSH cells.

After this cell completes, use the single Colab terminal only for:

```bash
ollama serve
```

Leave that terminal running. Run `ollama signin` from a notebook cell after the server is up.
"""))

!sudo apt-get update -y
!sudo apt-get install -y zstd curl

# DSH preview builds need a modern Node. Installing Node 24 is repeatable in Colab.
!curl -fsSL https://deb.nodesource.com/setup_24.x | sudo -E bash -
!sudo apt-get install -y nodejs

# Install Ollama CLI/runtime if it is missing.
!command -v ollama >/dev/null 2>&1 || curl -fsSL https://ollama.com/install.sh | sh

!node -v
!npm -v
!ollama --version


In [ ]:
# DeepSeek Harness + Ollama Cloud setup check.
# Prerequisite: `ollama serve` is running in the single Colab terminal.

import os
import shutil
from pathlib import Path
from getpass import getpass
from IPython.display import Markdown, display

DEEPSEEK_DSH_MODEL = os.environ.get("DEEPSEEK_DSH_MODEL", "deepseek-v4-flash:cloud")
os.environ["DEEPSEEK_DSH_MODEL"] = DEEPSEEK_DSH_MODEL

if not os.environ.get("OLLAMA_API_KEY"):
    try:
        from google.colab import userdata
        secret = userdata.get("OLLAMA_API_KEY")
    except Exception:
        secret = None
    os.environ["OLLAMA_API_KEY"] = secret or getpass("Enter OLLAMA_API_KEY: ")

# Ollama's generated DSH settings refer to this env var name.
os.environ["OLLAMA_LAUNCH_DSH_API_KEY"] = os.environ["OLLAMA_API_KEY"]

display(Markdown(f"""
## DeepSeek Harness Setup Check

Model selected for DSH through Ollama: `{DEEPSEEK_DSH_MODEL}`

This cell:

1. verifies local tools,
2. signs in / checks Ollama Cloud access,
3. asks `ollama launch dsh --config` to generate DSH settings,
4. copies those settings to `~/.dsh/settings.yaml` for direct headless `npx` runs.
"""))

!timeout 20s node --version || true
!timeout 20s npm --version || true
!timeout 20s ollama --version || true

if shutil.which("ollama") is None:
    raise RuntimeError("Ollama CLI is still missing. Re-run the runtime setup cell.")

# This requires `ollama serve` to be running in the Colab terminal.
!timeout 30s ollama list || true

# If this has not been done in the runtime yet, it will show the auth flow.
!ollama signin || true

# Configure DeepSeek Harness through Ollama without starting the Web UI.
!timeout 180s ollama launch dsh --model "$DEEPSEEK_DSH_MODEL" --config || true

ollama_settings = Path.home() / ".ollama" / "launch" / "dsh" / "settings.yaml"
dsh_settings = Path.home() / ".dsh" / "settings.yaml"
dsh_settings.parent.mkdir(parents=True, exist_ok=True)

if not ollama_settings.exists():
    raise RuntimeError(
        f"Ollama did not create {ollama_settings}. Check that ollama serve is running, signin completed, and the model is available."
    )

dsh_settings.write_text(ollama_settings.read_text())

print("Copied:", ollama_settings, "->", dsh_settings)
print("OLLAMA_API_KEY set:", bool(os.environ.get("OLLAMA_API_KEY")))
print("OLLAMA_LAUNCH_DSH_API_KEY set:", bool(os.environ.get("OLLAMA_LAUNCH_DSH_API_KEY")))
print("\nDSH settings now used by direct headless npx:")
print(dsh_settings.read_text())


In [ ]:
# DeepSeek Harness multi-agent smoke test.
# This verifies whether the current DSH preview/runtime can use blocking subagents in Colab.

import subprocess
from IPython.display import Markdown, display

DEEPSEEK_SUBAGENT_SMOKE_OUTPUT = ""
DEEPSEEK_SUBAGENT_SMOKE_ERROR = ""
DEEPSEEK_SUBAGENT_SMOKE_OK = False

smoke_prompt = (
    "You must test multi-agent delegation. Use the DSH subagent tool twice with run_in_background set to false. "
    "Subagent A must answer exactly: EVIDENCE_CHILD_OK. "
    "Subagent B must answer exactly: SAFETY_CHILD_OK. "
    "Wait for both child results. Then return a final answer with two lines: "
    "evidence_child=<exact child A answer> and safety_child=<exact child B answer>. "
    "If the subagent tool is unavailable or fails, report the exact failure text."
)

smoke_cmd = [
    "npx", "--yes", "@deepseek-ai/dsh",
    "--profile", "headless",
    smoke_prompt,
]

print("Running direct DSH headless blocking-subagent smoke test...")
print("Command:", " ".join(smoke_cmd[:-1]), "<smoke task text>")

try:
    smoke = subprocess.run(
        smoke_cmd,
        text=True,
        capture_output=True,
        timeout=240,
    )
    DEEPSEEK_SUBAGENT_SMOKE_OUTPUT = smoke.stdout.strip()
    DEEPSEEK_SUBAGENT_SMOKE_ERROR = smoke.stderr.strip()
    DEEPSEEK_SUBAGENT_SMOKE_OK = (
        "EVIDENCE_CHILD_OK" in DEEPSEEK_SUBAGENT_SMOKE_OUTPUT
        and "SAFETY_CHILD_OK" in DEEPSEEK_SUBAGENT_SMOKE_OUTPUT
    )
    print("Exit code:", smoke.returncode)
    print("Smoke test passed:", DEEPSEEK_SUBAGENT_SMOKE_OK)
    if DEEPSEEK_SUBAGENT_SMOKE_OUTPUT:
        display(Markdown("## DSH Subagent Smoke Test Stdout"))
        print(DEEPSEEK_SUBAGENT_SMOKE_OUTPUT)
    if DEEPSEEK_SUBAGENT_SMOKE_ERROR:
        display(Markdown("## DSH Subagent Smoke Test Stderr"))
        print(DEEPSEEK_SUBAGENT_SMOKE_ERROR)
except Exception as exc:
    DEEPSEEK_SUBAGENT_SMOKE_ERROR = repr(exc)
    display(Markdown("## DSH Subagent Smoke Test Did Not Complete"))
    print(DEEPSEEK_SUBAGENT_SMOKE_ERROR)


In [ ]:
# Attempt a bounded DeepSeek Harness multi-agent incident run through Ollama.
# This avoids the Web UI and requires blocking subagents before the lead agent produces the final plan.

import os
import subprocess
from pathlib import Path
from IPython.display import Markdown, display

DEEPSEEK_CAPTURED_OUTPUT = ""
DEEPSEEK_CAPTURED_ERROR = ""

dsh_workspace = Path("/content/dsh-incident-demo")
dsh_workspace.mkdir(exist_ok=True)

(dsh_workspace / "incident_ticket.md").write_text("""
Incident: INC-2026-08-17-042
Service: checkout-api
Detected: 2026-08-17T09:42:00Z
Impact: Customers see slow checkout and intermittent payment timeout errors.

Observed facts:
- p95 latency increased from 240ms to 2100ms.
- Logs show repeated promotion_price_cache miss events.
- Payment timeout errors appear downstream of checkout latency.
""".strip())

(dsh_workspace / "runbook.md").write_text("""
Runbook:
- Treat promotion_price_cache miss bursts as the likely first investigation path.
- Enable the promotion price cache single-flight lock.
- Lower promotion price cache TTL to 60 seconds during rollout.
- Keep payment writes enabled unless payment timeout error rate exceeds 12% for 5 minutes.
- Do not restart checkout pods unless there is crash-loop or memory-pressure evidence.
- Do not truncate or drop promotion cache tables during live traffic.
""".strip())

(dsh_workspace / "prior_incident.md").write_text("""
Prior lesson:
- Avoid restarting all checkout pods without crash-loop evidence.
- Previous incident was mitigated by single-flight cache miss protection plus short TTL during rollout.
""".strip())

prompt = (
    "You are the lead incident commander in a multi-agent DeepSeek Harness run. "
    "Read incident_ticket.md, runbook.md, and prior_incident.md. "
    "Before writing the final plan, use the DSH subagent tool with run_in_background set to false for two child agents. "
    "Child 1 role: Evidence Reviewer. It must read the files and report only the supported evidence and likely cause. "
    "Child 2 role: Safety Reviewer. It must read the files and report forbidden actions, allowed mitigations, and rollback constraints. "
    "Wait for both child results. In the final answer, include sections named exactly: "
    "Evidence Reviewer Subagent Report, Safety Reviewer Subagent Report, Final Incident Plan. "
    "The Final Incident Plan must include likely_cause, evidence, safe_next_action, rollback_plan, customer_impact, and open_questions. "
    "Do not invent tools, owners, dashboards, thresholds, or operational facts that are not in the files. "
    "If the subagent tool is unavailable or fails, report the exact failure and do not present the result as multi-agent."
)

cmd = [
    "npx", "--yes", "@deepseek-ai/dsh",
    "--profile", "headless",
    prompt,
]

print("Workspace prepared:", dsh_workspace)
print("Running multi-agent direct DSH headless with Ollama-configured model:", DEEPSEEK_DSH_MODEL)
print("Subagent smoke test passed:", globals().get("DEEPSEEK_SUBAGENT_SMOKE_OK"))
print("Command:", " ".join(cmd[:-1]), "<incident task text>")

try:
    completed = subprocess.run(
        cmd,
        cwd=str(dsh_workspace),
        text=True,
        capture_output=True,
        timeout=420,
    )
    DEEPSEEK_CAPTURED_OUTPUT = completed.stdout.strip()
    DEEPSEEK_CAPTURED_ERROR = completed.stderr.strip()
    print("Exit code:", completed.returncode)
    if DEEPSEEK_CAPTURED_OUTPUT:
        display(Markdown("## DeepSeek Harness Multi-Agent Headless Stdout"))
        print(DEEPSEEK_CAPTURED_OUTPUT)
    if DEEPSEEK_CAPTURED_ERROR:
        display(Markdown("## DeepSeek Harness Multi-Agent Headless Stderr"))
        print(DEEPSEEK_CAPTURED_ERROR)
except FileNotFoundError as exc:
    DEEPSEEK_CAPTURED_ERROR = str(exc)
    display(Markdown("## DeepSeek Harness Multi-Agent Headless Did Not Start"))
    print(DEEPSEEK_CAPTURED_ERROR)
except subprocess.TimeoutExpired as exc:
    DEEPSEEK_CAPTURED_OUTPUT = (exc.stdout or "").strip() if isinstance(exc.stdout, str) else ""
    DEEPSEEK_CAPTURED_ERROR = (exc.stderr or "").strip() if isinstance(exc.stderr, str) else "Timed out after 420 seconds"
    display(Markdown("## DeepSeek Harness Multi-Agent Headless Timed Out"))
    print(DEEPSEEK_CAPTURED_ERROR)


In [ ]:
# Score DeepSeek Harness multi-agent output, either captured from the previous cell or pasted manually.

DEEPSEEK_OUTPUT = (globals().get("DEEPSEEK_CAPTURED_OUTPUT") or "").strip()

if not DEEPSEEK_OUTPUT:
    DEEPSEEK_OUTPUT = """
Paste DeepSeek Harness incident-response output here if you run it separately.
""".strip()

if DEEPSEEK_OUTPUT and "Paste DeepSeek" not in DEEPSEEK_OUTPUT:
    deepseek_scored = score_freeform_answer(
        scenario=scenario,
        answer=DEEPSEEK_OUTPUT,
        lane=Lane.DEEPSEEK_PROVIDER,
        title="DeepSeek Harness multi-agent headless output experiment",
        takeaway="DeepSeek Harness represents the plug-and-play runtime direction for harness engineering.",
        used_harness_memory=True,
    )
    display(Markdown(render_management_summary_markdown(deepseek_scored)))
    display(Markdown(render_executive_findings_markdown(deepseek_scored)))
    display(Markdown(render_rule_findings_markdown(scenario, deepseek_scored)))
    display(HTML(render_model_output_html("DeepSeek Harness multi-agent headless output", deepseek_scored.final_answer)))

    deepseek_findings = evaluate_rules(scenario, deepseek_scored)
    deepseek_summary = summarize_findings_with_ollama(
        scenario=scenario,
        result=deepseek_scored,
        findings=deepseek_findings,
        model_name=SUMMARY_MODEL,
    )
    deepseek_management_root_cause = summarize_root_cause_for_management(
        scenario=scenario,
        result=deepseek_scored,
        findings=deepseek_findings,
        model_name=SUMMARY_MODEL,
    )
    deepseek_critique = critique_groundedness_with_ollama(
        scenario=scenario,
        result=deepseek_scored,
        model_name=SUMMARY_MODEL,
    )
    display(Markdown("# DeepSeek Harness: Management-Language Root Cause"))
    display(Markdown(deepseek_management_root_cause))
    display(Markdown("# DeepSeek Harness: LLM-Polished Rule Summary"))
    display(Markdown(deepseek_summary))
    display(Markdown("# DeepSeek Harness: Qualitative Groundedness Critique"))
    display(Markdown(deepseek_critique))
else:
    display(Markdown("""
## DeepSeek Harness Output Not Scored Yet

No DeepSeek Harness multi-agent headless output was captured or pasted in `DEEPSEEK_OUTPUT`.

For the management narrative, say:

> DeepSeek Harness shows the plug-and-play runtime direction. If the headless developer-preview path does not run cleanly in Colab today, that is a maturity/setup finding about the framework path, not a failure of harness engineering. The live Ollama no/weak/strong lanes already prove the harness value.
"""))


# Closing Narrative

The medium model is not magically smarter. It performs better because the harness gives it:

- controlled context
- tool boundaries
- shared memory
- policy/runbook grounding
- reviewer checks
- objective sensors
- repair loop
- repeatable scorecard

That is the difference between a chat answer and a production AI workflow.